# II. Nutrición de Cultivos y Fertilidad de Suelos

## Fundamento Agronómico y Matemático

### Modelo de Respuesta de Mitscherlich
Describe la relación entre dosis de fertilizante y rendimiento:

$$Y = Y_{max} \cdot \left(1 - e^{-c(x + b)}\right)$$

Donde $Y_{max}$ es el rendimiento máximo potencial, $c$ es el coeficiente de respuesta
y $b$ el nutriente disponible en el suelo.

### Kriging Ordinario (Geoestadística)
La varianza de predicción en un punto no muestreado $\mathbf{x_0}$:

$$\hat{Z}(\mathbf{x_0}) = \sum_{i=1}^{n} \lambda_i Z(\mathbf{x_i})$$

Con restricción $\sum \lambda_i = 1$ (insesgamiento). El variograma experimental:

$$\hat{\gamma}(h) = \frac{1}{2|N(h)|} \sum_{N(h)} [Z(\mathbf{x_i}) - Z(\mathbf{x_j})]^2$$

### Zonificación por Clustering (K-Means)
Se agrupan las muestras en zonas de manejo homogéneo minimizando:

$$J = \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit

def generar_datos_suelo(n_muestras=120, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1000, n_muestras)
    y = rng.uniform(0, 1200, n_muestras)
    trend_x = 0.003 * (x - 500)
    trend_y = 0.002 * (y - 600)
    mo = np.clip(3.5 + trend_x + rng.normal(0, 0.6, n_muestras), 1.0, 6.5)
    ph = np.clip(6.2 - 0.2 * mo + rng.normal(0, 0.25, n_muestras), 4.5, 8.0)
    p_bray = np.clip(15 + trend_y + rng.normal(0, 5, n_muestras), 3, 50)
    k_meq = np.clip(0.8 + rng.normal(0, 0.2, n_muestras), 0.2, 2.5)
    ce_ds = np.clip(0.5 + rng.normal(0, 0.15, n_muestras), 0.1, 2.0)
    rinde_hist = np.clip(
        4500 + 300 * (mo - 3.5) + 50 * (p_bray - 15) - 200 * np.abs(ph - 6.5) + rng.normal(0, 300, n_muestras),
        2000, 8000
    )
    return pd.DataFrame({
        'muestra_id': [f'M{i:04d}' for i in range(n_muestras)],
        'coord_x': np.round(x, 1),
        'coord_y': np.round(y, 1),
        'mo_pct': np.round(mo, 2),
        'ph': np.round(ph, 2),
        'p_bray_ppm': np.round(p_bray, 1),
        'k_meq_100g': np.round(k_meq, 2),
        'ce_ds_m': np.round(ce_ds, 2),
        'rinde_hist_kg_ha': np.round(rinde_hist, 0),
    })

print('Librerías cargadas.')

## Parámetros del Análisis

In [ ]:
# Parámetros fijos del análisis
n_zonas_val = 3           # Número de Zonas de Manejo (K-Means)
rinde_objetivo_val = 6000 # Rinde Objetivo (kg/ha)
precio_soja_val = 400     # Precio Soja (USD/t)
precio_urea_val = 600     # Precio Urea (USD/t)

print(f'Zonas K-Means: {n_zonas_val} | Rinde objetivo: {rinde_objetivo_val} kg/ha | '
      f'Precio soja: USD {precio_soja_val}/t | Precio urea: USD {precio_urea_val}/t')

In [ ]:
df_suelo = generar_datos_suelo(n_muestras=150)

features_cluster = ['mo_pct', 'p_bray_ppm', 'ph', 'k_meq_100g', 'ce_ds_m']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_suelo[features_cluster])
kmeans = KMeans(n_clusters=n_zonas_val, random_state=42, n_init=10)
df_suelo['zona'] = kmeans.fit_predict(X_scaled).astype(str)

print(f'Datos cargados: {len(df_suelo)} muestras | {n_zonas_val} zonas de manejo identificadas')

In [ ]:
paleta = px.colors.qualitative.Set2[:n_zonas_val]

fig_mapa = px.scatter(
    df_suelo, x='coord_x', y='coord_y', color='zona',
    size='mo_pct', hover_data=['ph', 'p_bray_ppm', 'k_meq_100g', 'rinde_hist_kg_ha'],
    title='Mapa de Zonificación de Suelo (K-Means)',
    labels={'coord_x': 'Este (m)', 'coord_y': 'Norte (m)', 'zona': 'Zona de Manejo'},
    color_discrete_sequence=paleta,
    template='plotly_white', height=450
)
fig_mapa.update_traces(marker=dict(opacity=0.85, line=dict(width=0.5, color='white')))
from IPython.display import display, HTML
display(HTML(fig_mapa.to_html(include_plotlyjs="cdn", full_html=False)))

stats_zona = df_suelo.groupby('zona')[['mo_pct', 'ph', 'p_bray_ppm', 'k_meq_100g', 'rinde_hist_kg_ha']].mean().round(2).reset_index()
stats_zona.columns = ['Zona', 'MO (%)', 'pH', 'P Bray (ppm)', 'K (meq)', 'Rinde hist. (kg/ha)']
stats_zona

## Curva de Respuesta de Mitscherlich para N

In [ ]:
dosis_n = np.linspace(0, 200, 100)
ymax = rinde_objetivo_val * 1.15
c_coef = 0.015
b_nativo = 40

rinde_resp = ymax * (1 - np.exp(-c_coef * (dosis_n + b_nativo)))
costo_n = dosis_n * 0.46 * (precio_urea_val / 1000)
ingreso = rinde_resp / 1000 * precio_soja_val
margen = ingreso - costo_n

dosis_optima = dosis_n[np.argmax(margen)]
margen_max = margen.max()

fig_curva = make_subplots(specs=[[{'secondary_y': True}]])
fig_curva.add_trace(go.Scatter(
    x=dosis_n, y=rinde_resp, name='Rinde (kg/ha)',
    line=dict(color='#27ae60', width=2.5)
))
fig_curva.add_trace(go.Scatter(
    x=dosis_n, y=margen, name='Margen bruto (USD/ha)',
    line=dict(color='#e67e22', width=2, dash='dash')
), secondary_y=True)
fig_curva.add_vline(x=dosis_optima, line_dash='dot', line_color='red',
                    annotation_text=f'Dosis óptima: {dosis_optima:.0f} kg N/ha')
fig_curva.update_layout(
    title=f'Curva de Respuesta Mitscherlich — Dosis óptima: {dosis_optima:.0f} kg N/ha | Margen: USD {margen_max:.0f}/ha',
    xaxis_title='Dosis de N (kg/ha)', template='plotly_white', height=420
)
fig_curva.update_yaxes(title_text='Rinde (kg/ha)', secondary_y=False)
fig_curva.update_yaxes(title_text='Margen Bruto (USD/ha)', secondary_y=True)
from IPython.display import display, HTML
display(HTML(fig_curva.to_html(include_plotlyjs="cdn", full_html=False)))

## Variograma Experimental de Materia Orgánica

In [ ]:
n = len(df_suelo)
distancias = []
semivarianzas = []
bins = np.arange(0, 500, 40)

for i in range(n):
    for j in range(i + 1, n):
        d = np.sqrt((df_suelo['coord_x'].iloc[i] - df_suelo['coord_x'].iloc[j])**2 +
                    (df_suelo['coord_y'].iloc[i] - df_suelo['coord_y'].iloc[j])**2)
        sv = 0.5 * (df_suelo['mo_pct'].iloc[i] - df_suelo['mo_pct'].iloc[j])**2
        distancias.append(d)
        semivarianzas.append(sv)

dist_arr = np.array(distancias)
sv_arr = np.array(semivarianzas)

bin_centers, bin_sv, bin_n = [], [], []
for k in range(len(bins) - 1):
    mask = (dist_arr >= bins[k]) & (dist_arr < bins[k + 1])
    if mask.sum() > 5:
        bin_centers.append((bins[k] + bins[k + 1]) / 2)
        bin_sv.append(sv_arr[mask].mean())
        bin_n.append(mask.sum())

def variograma_exponencial(h, nugget, sill, rango):
    return nugget + (sill - nugget) * (1 - np.exp(-h / rango))

try:
    popt_v, _ = curve_fit(
        variograma_exponencial, bin_centers, bin_sv,
        p0=[0.05, 0.3, 200], bounds=(0, [0.5, 1.0, 1000])
    )
    h_fit = np.linspace(0, max(bin_centers), 200)
    sv_fit = variograma_exponencial(h_fit, *popt_v)
    nugget_v, sill_v, rango_v = popt_v
except Exception:
    h_fit, sv_fit = [], []
    nugget_v, sill_v, rango_v = 0, 0, 0

fig_vario = go.Figure()
fig_vario.add_trace(go.Scatter(
    x=bin_centers, y=bin_sv, mode='markers+text',
    text=[f'n={v}' for v in bin_n],
    textposition='top center',
    name='γ̂(h) experimental', marker=dict(color='#3498db', size=10)
))
if len(h_fit) > 0:
    fig_vario.add_trace(go.Scatter(
        x=h_fit, y=sv_fit, mode='lines',
        name=f'Modelo Exp. — Nugget={nugget_v:.3f}, Sill={sill_v:.3f}, Rango={rango_v:.0f}m',
        line=dict(color='#e74c3c', width=2)
    ))
    fig_vario.add_hline(y=sill_v, line_dash='dot', line_color='gray',
                        annotation_text=f'Sill={sill_v:.3f}')

fig_vario.update_layout(
    title='Variograma Experimental de MO (%) — Base para Kriging',
    xaxis_title='Distancia de separación (m)',
    yaxis_title='Semivarianza γ̂(h)',
    template='plotly_white', height=400
)
from IPython.display import display, HTML
display(HTML(fig_vario.to_html(include_plotlyjs="cdn", full_html=False)))

print(f'Parámetros del variograma: Nugget = {nugget_v:.3f} | Sill = {sill_v:.3f} | Rango = {rango_v:.0f} m')
print(f'El rango indica que muestras separadas por más de {rango_v:.0f}m son espacialmente independientes.')